In [13]:
features = features.rename(
    columns={"future_nino34": "target"}
)

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import matplotlib.pyplot as plt

In [2]:
ROOT = Path("..")

PROCESSED_DATA = ROOT / "data" / "processed"

features = pd.read_csv(
    PROCESSED_DATA / "enso_features.csv",
    parse_dates=["Date"]
)

features.head()

,Date,nino34,nino3,nino4,iod,soi,nino34_lag1,nino34_lag2,nino34_lag3,nino3_lag1,...,iod_lag3,soi_lag1,soi_lag2,soi_lag3,nino34_roll3,nino3_roll3,nino4_roll3,iod_roll3,soi_roll3,future_nino34
0,1951-04-01,-0.23,-0.21,-0.42,-0.513,-0.3,-0.38,-1.04,-1.30,-0.33,...,0.256,-0.1,0.9,1.5,-0.550000,-0.433333,-0.703333,-0.014333,0.166667,-0.01
1,1951-05-01,-0.01,-0.18,0.26,-0.138,-0.7,-0.23,-0.38,-1.04,-0.21,...,0.211,-0.3,-0.1,0.9,-0.206667,-0.240000,-0.246667,-0.130667,-0.366667,0.00
2,1951-06-01,0.00,0.04,0.08,-0.190,0.2,-0.01,-0.23,-0.38,-0.18,...,0.259,-0.7,-0.3,-0.1,-0.080000,-0.116667,-0.026667,-0.280333,-0.266667,0.30
3,1951-07-01,0.30,0.62,0.23,-0.220,-1.0,0.00,-0.01,-0.23,0.04,...,-0.513,0.2,-0.7,-0.3,0.096667,0.160000,0.190000,-0.182667,-0.500000,0.17
4,1951-08-01,0.17,0.41,-0.26,0.124,-0.2,0.30,0.00,-0.01,0.62,...,-0.138,-1.0,0.2,-0.7,0.156667,0.356667,0.016667,-0.095333,-0.333333,0.51


In [3]:
X = features.drop(
    columns=[
        "Date",
        "future_nino34"
    ]
)

y = features["future_nino34"]

print(X.shape)
print(y.shape)

(887, 25)
(887,)


In [4]:
#train-test split
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):

    print(f"Fold {fold}")

    print(
        len(train_idx),
        len(test_idx)
    )

Fold 1
152 147
Fold 2
299 147
Fold 3
446 147
Fold 4
593 147
Fold 5
740 147


In [5]:
baseline = features["nino34"]

actual = features["future_nino34"]

mae = mean_absolute_error(actual, baseline)

rmse = np.sqrt(
    mean_squared_error(actual, baseline)
)

corr = np.corrcoef(
    actual,
    baseline
)[0,1]

print("Persistence Model")

print("MAE :", mae)

print("RMSE:", rmse)

print("Correlation:", corr)

Persistence Model
MAE : 0.20994363021420517
RMSE: 0.266731014408431
Correlation: 0.9530419924118501


In [8]:
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

evaluate_model(
    rf_model,
    X,
    y,
    "Random Forest"
)

Random Forest
Average MAE  : 0.2073
Average RMSE : 0.2664
Average Corr : 0.9509


In [6]:
def evaluate_model(model, X, y, model_name):

    tscv = TimeSeriesSplit(n_splits=5)

    mae_scores = []
    rmse_scores = []
    corr_scores = []

    for train_idx, test_idx in tscv.split(X):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        mae_scores.append(
            mean_absolute_error(y_test, predictions)
        )

        rmse_scores.append(
            np.sqrt(mean_squared_error(y_test, predictions))
        )

        corr_scores.append(
            np.corrcoef(y_test, predictions)[0,1]
        )

    print("="*60)
    print(model_name)
    print("="*60)

    print(f"Average MAE  : {np.mean(mae_scores):.4f}")
    print(f"Average RMSE : {np.mean(rmse_scores):.4f}")
    print(f"Average Corr : {np.mean(corr_scores):.4f}")

In [7]:
linear_model = LinearRegression()

evaluate_model(
    linear_model,
    X,
    y,
    "Linear Regression"
)

Linear Regression
Average MAE  : 0.1724
Average RMSE : 0.2192
Average Corr : 0.9667


In [14]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate_arimax(data):

    maes = []
    rmses = []
    corrs = []

    tscv = TimeSeriesSplit(n_splits=5)

    exog_cols = [
        "nino3",
        "nino4",
        "iod",
        "soi"
    ]

    for train_index, test_index in tscv.split(data):

        train = data.iloc[train_index]
        test = data.iloc[test_index]

        model = SARIMAX(
            train["target"],
            exog=train[exog_cols],
            order=(3,0,0),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        model_fit = model.fit(disp=False)

        predictions = model_fit.predict(
            start=test.index[0],
            end=test.index[-1],
            exog=test[exog_cols]
        )

        mae = mean_absolute_error(test["target"], predictions)

        rmse = np.sqrt(
            mean_squared_error(test["target"], predictions)
        )

        corr = np.corrcoef(
            test["target"],
            predictions
        )[0,1]

        maes.append(mae)
        rmses.append(rmse)
        corrs.append(corr)

    print("="*60)
    print("ARIMAX")
    print("="*60)

    print("Average MAE  :", round(np.mean(maes),4))
    print("Average RMSE :", round(np.mean(rmses),4))
    print("Average Corr :", round(np.mean(corrs),4))
evaluate_arimax(features)

ARIMAX
Average MAE  : 0.4226
Average RMSE : 0.5193
Average Corr : 0.6748


In [25]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate_arimax(data):

    maes = []
    rmses = []
    corrs = []

    tscv = TimeSeriesSplit(n_splits=5)

    exog_cols = [

        "nino3",
        "nino4",
        "iod",
        "soi",

        "nino34_roll3",
        "nino3_roll3",
        "nino4_roll3",
        "iod_roll3",
        "soi_roll3"
    ]

    candidate_orders = [
        (1,0,0),
        (2,0,0),
        (3,0,0),
        (1,0,1),
        (2,0,1),
        (3,0,1)
    ]

    for fold, (train_index, test_index) in enumerate(tscv.split(data), start=1):

        train = data.iloc[train_index]
        test = data.iloc[test_index]

        best_model = None
        best_aic = np.inf
        best_order = None

        for order in candidate_orders:

            try:

                model = SARIMAX(
                    train["target"],
                    exog=train[exog_cols],
                    order=order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )

                fitted = model.fit(disp=False)

                if fitted.aic < best_aic:
                    best_aic = fitted.aic
                    best_model = fitted
                    best_order = order

            except Exception as e:
                print(f"Fold {fold} Order {order} failed: {e}")

        if best_model is None:
            print(f"No model fitted for Fold {fold}")
            continue

        print(f"Fold {fold} Best Order: {best_order}")

        predictions = best_model.predict(
            start=test.index[0],
            end=test.index[-1],
            exog=test[exog_cols]
        )

        mae = mean_absolute_error(
            test["target"],
            predictions
        )

        rmse = np.sqrt(
            mean_squared_error(
                test["target"],
                predictions
            )
        )

        corr = np.corrcoef(
            test["target"],
            predictions
        )[0,1]

        maes.append(mae)
        rmses.append(rmse)
        corrs.append(corr)

    print("\n" + "="*60)
    print("ARIMAX (Tuned)")
    print("="*60)
    print("Average MAE  :", round(np.mean(maes),4))
    print("Average RMSE :", round(np.mean(rmses),4))
    print("Average Corr :", round(np.mean(corrs),4))

evaluate_arimax(features)

/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

Fold 1 Best Order: (3, 0, 1)


/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

Fold 2 Best Order: (2, 0, 1)


/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

...

/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

Fold 4 Best Order: (2, 0, 1)


/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

Fold 5 Best Order: (3, 0, 1)

ARIMAX (Tuned)
Average MAE  : 0.2898
Average RMSE : 0.3753
Average Corr : 0.9171


/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
